[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danilo-ds7/licitaciones-text-mining/blob/main/TextMining_Transformers_Licitaciones.ipynb)

# Text mining + Transformers sobre licitaciones reales de Mercado Público
**Módulo V — Arquitecturas de Atención y Transformers para la Gestión de Complejidad**
Docente: Danilo Gómez Correa

En este notebook aplicamos el flujo visto en clase sobre **5.021 licitaciones reales** publicadas en Mercado Público (ChileCompra) entre el 1 y el 6 de octubre de 2026:

| Etapa | Pregunta | Técnicas |
|---|---|---|
| **1. Exploración clásica** | ¿Qué dice el texto, a simple vista? | Nube de palabras, frecuencias y TF-IDF, n-gramas, red de co-ocurrencia, temas con LDA |
| **2. Visualizaciones con Transformers** | ¿Qué significa? | Embeddings + UMAP, BERTopic, mapa de atención, extracción estructurada y dashboard |

**Fuente de datos:** API pública de Mercado Público (`api.mercadopublico.cl`), consultada el 7 de octubre de 2026. Se incluye el listado completo de esos días (código, nombre, estado, fecha de cierre) y el detalle completo (descripción, organismo, región, montos, ítems) de una muestra aleatoria de 213 licitaciones.

> Ejecuta las celdas en orden con **Entorno de ejecución → Ejecutar todas**. No requiere GPU.

## 0. Preparación del entorno

In [ ]:
import sys, os
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    !pip -q install wordcloud umap-learn bertopic sentence-transformers networkx plotly google-genai anthropic
print('Entorno Colab' if EN_COLAB else 'Entorno local')

In [ ]:
import re, unicodedata, json, itertools, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

pd.set_option('display.max_colwidth', 120)
plt.rcParams['figure.dpi'] = 110
SEMILLA = 42

## 1. Cargar los datos
Los datos están en el repositorio del curso en GitHub. Si quieres descargar licitaciones de **otros días**, al final del notebook (Anexo A) está el código para consultar la API con tu propio *ticket*.

In [ ]:
REPO = 'https://raw.githubusercontent.com/danilo-ds7/licitaciones-text-mining/main/data'
BASE = 'data' if os.path.exists('data/licitaciones.csv') else REPO

df = pd.read_csv(f'{BASE}/licitaciones.csv')
print(df.shape)
df[['codigo', 'nombre', 'estado', 'tipo', 'fecha_cierre']].head(10)

In [ ]:
# ¿Cuántas licitaciones tienen detalle completo (descripción, organismo, montos)?
print('Con detalle:', df['tiene_detalle'].sum(), 'de', len(df))
df['estado'].value_counts()

## 2. Limpieza del texto
Antes de contar palabras hay que **normalizar**: minúsculas, sin tildes, sin números y sin *stopwords* (palabras vacías como "de", "la", "para").
Además quitamos palabras del **dominio** que aparecen en casi todas las licitaciones ("adquisición", "servicio", "suministro") y no ayudan a distinguirlas.

In [ ]:
STOPWORDS_ES = set('''a al algo algunas algunos ante antes como con contra cual cuales cuando de del desde donde durante e el ella ellas
ellos en entre era es esa esas ese eso esos esta estas este esto estos fue fueron ha han hasta la las le les lo los mas me mi mis mucho
muy ni no nos o otra otras otro otros para pero poco por porque que quien se sea segun ser si sin sobre su sus tambien tan te tiene
tienen todo todos tu tus un una unas uno unos y ya cada mediante traves vez via etc n nº n° nro num'''.split())

STOPWORDS_DOMINIO = set('''adquisicion adquisiciones compra compras contratacion servicio servicios suministro licitacion licitaciones
publica publico segun bases convenio año ano anos años 2026 2025 2027 unidad unidades varios varias tipo general generales municipalidad
municipal comuna region hospital departamento direccion adq serv san santa ppta solic'''.split())

def quitar_tildes(t):
    return ''.join(c for c in unicodedata.normalize('NFD', t) if unicodedata.category(c) != 'Mn')

SW = {quitar_tildes(w) for w in STOPWORDS_ES | STOPWORDS_DOMINIO}

def limpiar(texto):
    t = quitar_tildes(str(texto).lower())
    t = re.sub(r'[^a-zñ\s]', ' ', t)
    return ' '.join(w for w in t.split() if len(w) > 2 and w not in SW)

df['nombre_limpio'] = df['nombre'].fillna('').map(limpiar)
df[['nombre', 'nombre_limpio']].sample(8, random_state=SEMILLA)

---
# PARTE 1 · Exploración clásica: qué dice el texto, a simple vista
### 1.1 Nube de palabras
Términos frecuentes en los nombres de las licitaciones. Es llamativa, pero **no permite comparar magnitudes** con precisión.

In [ ]:
texto_total = ' '.join(df['nombre_limpio'])
nube = WordCloud(width=1400, height=650, background_color='white', colormap='viridis',
                 max_words=150, random_state=SEMILLA, collocations=False).generate(texto_total)
plt.figure(figsize=(13, 6))
plt.imshow(nube, interpolation='bilinear'); plt.axis('off')
plt.title('Nube de palabras: nombres de 5.021 licitaciones (1-6 oct 2026)')
plt.show()

### 1.2 Barras de frecuencia y TF-IDF
La versión **cuantitativa** de la nube. Conviene mostrarlas juntas.
- **Frecuencia:** cuántas veces aparece la palabra en total.
- **TF-IDF:** pondera la frecuencia por qué tan *específica* es la palabra (penaliza las que aparecen en casi todos los documentos).

In [ ]:
cv = CountVectorizer(min_df=3)
Xc = cv.fit_transform(df['nombre_limpio'])
frec = pd.Series(np.asarray(Xc.sum(axis=0)).ravel(), index=cv.get_feature_names_out()).sort_values(ascending=False)

tfidf = TfidfVectorizer(min_df=3)
Xt = tfidf.fit_transform(df['nombre_limpio'])
peso_tfidf = pd.Series(np.asarray(Xt.mean(axis=0)).ravel(), index=tfidf.get_feature_names_out()).sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(13, 6))
frec.head(20)[::-1].plot.barh(ax=ax[0], color='#091D44'); ax[0].set_title('Top 20 por frecuencia')
peso_tfidf.head(20)[::-1].plot.barh(ax=ax[1], color='#D68014'); ax[1].set_title('Top 20 por TF-IDF promedio')
plt.tight_layout(); plt.show()

**Pregunta:** ¿qué palabras cambian de posición entre ambos rankings? ¿Por qué?

### 1.3 N-gramas
Frases que se repiten. Un **bigrama** son 2 palabras seguidas; un **trigrama**, 3. Aquí aparecen expresiones como "mantención preventiva" o "artículos de aseo".
Para n-gramas **no** quitamos las palabras de dominio, porque forman parte de la frase.

In [ ]:
def limpiar_suave(texto):
    t = quitar_tildes(str(texto).lower())
    t = re.sub(r'[^a-zñ\s]', ' ', t)
    sw = {quitar_tildes(w) for w in STOPWORDS_ES}
    return ' '.join(w for w in t.split() if len(w) > 2 and w not in sw)

df['nombre_suave'] = df['nombre'].fillna('').map(limpiar_suave)

def top_ngramas(textos, n, k=15):
    v = CountVectorizer(ngram_range=(n, n), min_df=3)
    X = v.fit_transform(textos)
    return pd.Series(np.asarray(X.sum(axis=0)).ravel(), index=v.get_feature_names_out()).sort_values(ascending=False).head(k)

bi, tri = top_ngramas(df['nombre_suave'], 2), top_ngramas(df['nombre_suave'], 3)
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
bi[::-1].plot.barh(ax=ax[0], color='#228054'); ax[0].set_title('Bigramas más frecuentes')
tri[::-1].plot.barh(ax=ax[1], color='#3C78BE'); ax[1].set_title('Trigramas más frecuentes')
plt.tight_layout(); plt.show()

### 1.4 Red de co-ocurrencia
¿Qué palabras aparecen **juntas** en el mismo nombre de licitación? Cada nodo es una palabra; el grosor de la línea, cuántas veces aparecen juntas.

In [ ]:
top_palabras = set(frec.head(45).index)
pares = collections.Counter()
for t in df['nombre_limpio']:
    ws = sorted(set(t.split()) & top_palabras)
    pares.update(itertools.combinations(ws, 2))

G = nx.Graph()
for (a, b), w in pares.most_common(70):
    G.add_edge(a, b, weight=w)
pos = nx.spring_layout(G, k=0.6, seed=SEMILLA)

plt.figure(figsize=(12, 9))
anchos = [G[u][v]['weight'] / max(nx.get_edge_attributes(G, 'weight').values()) * 6 for u, v in G.edges()]
nx.draw_networkx_edges(G, pos, width=anchos, alpha=0.35, edge_color='#091D44')
tam = [frec[n] * 3 for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_size=tam, node_color='#D68014', alpha=0.85)
nx.draw_networkx_labels(G, pos, font_size=9)
plt.title('Red de co-ocurrencia de palabras (top 70 pares)'); plt.axis('off'); plt.show()

pd.DataFrame(pares.most_common(10), columns=['par', 'veces'])

### 1.5 Temas con LDA
**LDA** (*Latent Dirichlet Allocation*) supone que cada documento es una mezcla de temas, y cada tema una mezcla de palabras. Solo mira **conteos de palabras**.

In [ ]:
N_TEMAS = 8
lda = LatentDirichletAllocation(n_components=N_TEMAS, random_state=SEMILLA, learning_method='batch', max_iter=25)
doc_tema = lda.fit_transform(Xc)
vocab = cv.get_feature_names_out()
temas_lda = {k: ', '.join(vocab[i] for i in lda.components_[k].argsort()[::-1][:7]) for k in range(N_TEMAS)}
df['tema_lda'] = doc_tema.argmax(axis=1)
resumen_lda = (df['tema_lda'].value_counts().rename('n').to_frame()
               .assign(palabras=lambda d: d.index.map(temas_lda)).sort_index())
resumen_lda

In [ ]:
# Ejemplos de licitaciones por tema LDA
for k in range(3):
    print(f'Tema {k}: {temas_lda[k]}')
    for n in df.loc[df['tema_lda'] == k, 'nombre'].sample(3, random_state=SEMILLA):
        print('   -', n)

**Pregunta:** ¿los temas de LDA son fáciles de interpretar? ¿Hay temas que mezclan cosas distintas?

---
# PARTE 2 · Visualizaciones con Transformers: qué significa
### 2.1 Embeddings con un Transformer
Usamos **`paraphrase-multilingual-MiniLM-L12-v2`**, un Transformer de 12 capas (tipo BERT, solo codificador) entrenado para que frases con **el mismo significado** queden cerca, en más de 50 idiomas, incluido el español.
Cada nombre de licitación se convierte en un vector de **384 números**.

> Para ahorrar tiempo, el repositorio trae los embeddings ya calculados con este mismo modelo. Cambia `USAR_PRECALCULADOS = False` para calcularlos tú mismo (en Colab tarda 1 a 2 minutos con CPU).

In [ ]:
MODELO = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
USAR_PRECALCULADOS = True

modelo_st = None
def cargar_modelo():
    global modelo_st
    if modelo_st is None:
        from sentence_transformers import SentenceTransformer
        modelo_st = SentenceTransformer(MODELO)
    return modelo_st

if USAR_PRECALCULADOS:
    import io, urllib.request
    ruta = f'{BASE}/embeddings.npy'
    E = np.load(ruta) if not ruta.startswith('http') else np.load(io.BytesIO(urllib.request.urlopen(ruta).read()))
    E = E.astype(np.float32)
else:
    E = cargar_modelo().encode(df['nombre'].fillna('').tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
print('Matriz de embeddings:', E.shape)

#### ¿"Falla" y "avería" son lo mismo?
El conteo de palabras dice que **no** (no comparten letras). Los embeddings dicen que **sí**. Medimos la similitud coseno entre frases.

In [ ]:
frases = ['reparación de falla en bomba de agua', 'arreglo de avería en equipo de bombeo',
          'compra de cemento de alta resistencia', 'adquisición de artículos de aseo']
try:
    Ef = cargar_modelo().encode(frases, normalize_embeddings=True)
    sim_emb = pd.DataFrame(Ef @ Ef.T, index=frases, columns=frases).round(2)
except Exception as e:
    print('No se pudo descargar el modelo en este entorno (en Colab sí funciona):', e)
    sim_emb = None

tv = TfidfVectorizer().fit(frases)
T = tv.transform(frases).toarray(); T = T / np.linalg.norm(T, axis=1, keepdims=True)
sim_tfidf = pd.DataFrame(T @ T.T, index=frases, columns=frases).round(2)
print('Similitud con TF-IDF (conteo de palabras):'); display(sim_tfidf)
if sim_emb is not None:
    print('Similitud con embeddings (Transformer):'); display(sim_emb)

### 2.2 Búsqueda semántica: licitaciones parecidas
Elegimos una licitación y buscamos las más parecidas, primero con **TF-IDF** y luego con **embeddings**. Observa las que aparecen con embeddings aunque no compartan palabras.

In [ ]:
def similares(i, M, k=6):
    s = M @ M[i]
    orden = np.argsort(-s)
    orden = [j for j in orden if j != i][:k]
    return pd.DataFrame({'nombre': df['nombre'].iloc[orden].values, 'similitud': s[orden].round(3)})

Xt_n = Xt.toarray() if hasattr(Xt, 'toarray') else Xt
Xt_n = Xt_n / np.clip(np.linalg.norm(Xt_n, axis=1, keepdims=True), 1e-9, None)

i = int(df.index[df['nombre'].str.contains('MANTENCI', case=False, na=False) &
                 df['nombre'].str.contains('ASCENSOR|CALDERA|GENERADOR|BOMBA', case=False, na=False)][0])
print('Consulta:', df['nombre'].iloc[i])
print('\nCon TF-IDF:');      display(similares(i, Xt_n))
print('Con embeddings:');    display(similares(i, E))

### 2.3 Mapa de embeddings (UMAP)
Reducimos los 384 números de cada licitación a **2 dimensiones** con UMAP. Cada punto es una licitación; las parecidas quedan **juntas aunque usen palabras distintas**. Pasa el mouse sobre los puntos.

In [ ]:
import umap
reductor = umap.UMAP(n_neighbors=15, n_components=2, min_dist=0.08, metric='cosine', random_state=SEMILLA)
XY = reductor.fit_transform(E)
df['x'], df['y'] = XY[:, 0], XY[:, 1]
fig = px.scatter(df, x='x', y='y', color=df['tema_lda'].astype(str), hover_name='nombre',
                 hover_data={'x': False, 'y': False, 'estado': True},
                 title='Mapa de embeddings (color = tema LDA)', opacity=0.7, height=620,
                 labels={'color': 'Tema LDA'})
fig.update_traces(marker_size=5); fig.show()

**Pregunta:** ¿los colores de LDA coinciden con los grupos que forma el mapa? ¿Qué te dice eso de LDA frente a los embeddings?

### 2.4 Clusters semánticos con BERTopic
**BERTopic** agrupa los embeddings (UMAP + HDBSCAN) y luego describe cada grupo con sus palabras más representativas (c-TF-IDF). Suele dar temas **más coherentes que LDA**, porque agrupa por significado y no por conteo.

In [ ]:
from bertopic import BERTopic
from hdbscan import HDBSCAN

docs_limpios = df['nombre_limpio'].tolist()
modelo_temas = BERTopic(
    umap_model=umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=SEMILLA),
    hdbscan_model=HDBSCAN(min_cluster_size=25, min_samples=10, prediction_data=True),
    vectorizer_model=CountVectorizer(ngram_range=(1, 2), min_df=3),
    language='multilingual')
temas, _ = modelo_temas.fit_transform(docs_limpios, embeddings=E)
df['tema_bertopic'] = temas
info = modelo_temas.get_topic_info()
print('Temas encontrados:', len(info) - 1, '| Sin tema (ruido):', (np.array(temas) == -1).sum())
info[['Topic', 'Count', 'Name']].head(20)

In [ ]:
etiquetas = {r.Topic: ('Sin tema' if r.Topic == -1 else ', '.join(w for w, _ in modelo_temas.get_topic(r.Topic)[:3]))
             for r in info.itertuples()}
df['tema_bertopic_nombre'] = df['tema_bertopic'].map(etiquetas)
plot_df = df[df['tema_bertopic'] != -1]
fig = px.scatter(plot_df, x='x', y='y', color='tema_bertopic_nombre', hover_name='nombre',
                 hover_data={'x': False, 'y': False}, title='Mapa de embeddings (color = tema BERTopic)',
                 opacity=0.75, height=650, labels={'tema_bertopic_nombre': 'Tema'})
fig.update_traces(marker_size=5); fig.show()

In [ ]:
# Barras de palabras clave por tema (gráfico propio de BERTopic)
modelo_temas.visualize_barchart(top_n_topics=8, n_words=6)

### 2.5 Mapa de atención
¿En qué palabras se fija el Transformer al representar una licitación? Extraemos la **matriz de atención** de la última capa del mismo modelo (promedio de sus 12 cabezas).
Recuerda: la atención muestra **qué mira** el modelo, no es por sí sola una explicación causal.

In [ ]:
def mapa_atencion(texto, capa=-1):
    import torch
    from transformers import AutoTokenizer, AutoModel
    tok = AutoTokenizer.from_pretrained(MODELO)
    mod = AutoModel.from_pretrained(MODELO, output_attentions=True, attn_implementation='eager')
    enc = tok(texto, return_tensors='pt')
    with torch.no_grad():
        att = mod(**enc).attentions[capa][0].mean(0).numpy()   # (tokens, tokens), promedio de cabezas
    tokens = tok.convert_ids_to_tokens(enc['input_ids'][0])
    return att, tokens

def graficar_atencion(att, tokens, titulo):
    fig = px.imshow(att, x=tokens, y=tokens, color_continuous_scale='Blues', aspect='auto',
                    labels=dict(x='token consultado', y='token que consulta', color='peso'), title=titulo)
    fig.update_layout(height=560); fig.show()

texto = 'Mantención preventiva y correctiva de bombas por falla de rodamientos'
try:
    att, tokens = mapa_atencion(texto)
    graficar_atencion(att, tokens, f'Atención (última capa): "{texto}"')
except Exception as e:
    print('No se pudo descargar el modelo en este entorno (en Colab sí funciona):', e)

**Pregunta:** ¿qué tokens reciben más atención desde "falla"? ¿Cómo se dividió la palabra "rodamientos" en tokens?

### 2.6 Extracción estructurada: del texto a un registro
Tomamos las licitaciones **con detalle** y convertimos su descripción en campos. Primero con **reglas (expresiones regulares)**, que es text mining clásico; luego, opcionalmente, con un **LLM** que llena un esquema JSON.

In [ ]:
det = df[df['tiene_detalle']].copy()
print(len(det), 'licitaciones con descripción')
det[['codigo', 'nombre', 'descripcion', 'organismo', 'region', 'monto_estimado']].head(5)

In [ ]:
# Extracción con reglas: plazo en meses/días y presencia de cláusulas típicas
def extraer_reglas(t):
    t = str(t).lower()
    m = re.search(r'(\d{1,3})\s*(meses|mes)\b', t)
    d = re.search(r'(\d{1,3})\s*(d[ií]as)\b', t)
    return pd.Series({
        'plazo_meses': int(m.group(1)) if m else None,
        'plazo_dias': int(d.group(1)) if d else None,
        'menciona_garantia': bool(re.search(r'garant[ií]a', t)),
        'menciona_multa': bool(re.search(r'multa|penalidad|sanci[oó]n', t)),
        'menciona_mantencion': bool(re.search(r'mantenci[oó]n|mantenimiento', t)),
    })

texto_det = det['descripcion'].fillna('') + ' ' + det['items_descripcion'].fillna('')
det = det.join(texto_det.apply(extraer_reglas))
det[['nombre', 'plazo_meses', 'plazo_dias', 'menciona_garantia', 'menciona_multa', 'menciona_mantencion']].head(10)

#### Extracción con LLM (opcional)
Con un LLM se pide completar un **esquema fijo** y responder solo JSON. Usa tu propia clave de API guardada en los *Secretos* de Colab (ícono de llave a la izquierda) con el nombre `GEMINI_API_KEY` (Gemini tiene un nivel gratuito) o `ANTHROPIC_API_KEY`.
Si no hay clave, la celda se omite.

In [ ]:
ESQUEMA = '''Responde SOLO con un JSON con estas claves:
{"rubro": str, "bien_o_servicio": "bien"|"servicio"|"obra", "plazo_meses": int|null,
 "requiere_visita_terreno": bool|null, "palabras_clave": [str, str, str]}
Si un dato no aparece en el texto, usa null. No inventes valores.'''

def clave(nombre):
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        return os.environ.get(nombre)

def extraer_llm(texto):
    if clave('GEMINI_API_KEY'):
        from google import genai
        cli = genai.Client(api_key=clave('GEMINI_API_KEY'))
        r = cli.models.generate_content(model='gemini-2.5-flash', contents=f'{ESQUEMA}\n\nTexto:\n{texto}')
        salida = r.text
    elif clave('ANTHROPIC_API_KEY'):
        import anthropic
        cli = anthropic.Anthropic(api_key=clave('ANTHROPIC_API_KEY'))
        r = cli.messages.create(model='claude-haiku-4-5', max_tokens=400,
                                messages=[{'role': 'user', 'content': f'{ESQUEMA}\n\nTexto:\n{texto}'}])
        salida = r.content[0].text
    else:
        return None
    return json.loads(re.search(r'\{.*\}', salida, re.S).group(0))

registros = []
for _, fila in det.head(10).iterrows():
    try:
        r = extraer_llm(f"{fila['nombre']}. {fila['descripcion']}")
    except Exception as e:
        r = {'error': str(e)[:80]}
    if r is None:
        print('Sin clave de API: se omite la extracción con LLM.'); break
    registros.append({'codigo': fila['codigo'], **r})
pd.DataFrame(registros) if registros else None

### 2.7 Dashboard de lo extraído
Con los campos ya estructurados (los que entrega la API y los que extrajimos) armamos indicadores para la gestión.

In [ ]:
TIPOS = {'L1': '< 100 UTM', 'LE': '100-1.000 UTM', 'LP': '1.000-2.000 UTM', 'LQ': '2.000-5.000 UTM', 'LR': '> 5.000 UTM'}
tipos = df['tipo'].map(TIPOS).fillna('Otros (privadas, obras, etc.)').value_counts()
fig = px.bar(x=tipos.values, y=tipos.index, orientation='h', title='Licitaciones por tramo de monto (según código)',
             labels={'x': 'N.º de licitaciones', 'y': ''}, color_discrete_sequence=['#091D44'])
fig.update_layout(height=380, yaxis={'categoryorder': 'total ascending'}); fig.show()

In [ ]:
det['fecha_publicacion'] = pd.to_datetime(det['fecha_publicacion'], errors='coerce')
det['fecha_cierre'] = pd.to_datetime(det['fecha_cierre'], errors='coerce')
det['dias_para_ofertar'] = (det['fecha_cierre'] - det['fecha_publicacion']).dt.days

reg = det['region'].replace('', np.nan).dropna().str.replace('Región ', '', regex=False).value_counts().head(12)
fig = px.bar(x=reg.values, y=reg.index, orientation='h', title='Licitaciones con detalle por región (muestra)',
             labels={'x': 'N.º', 'y': ''}, color_discrete_sequence=['#D68014'])
fig.update_layout(height=420, yaxis={'categoryorder': 'total ascending'}); fig.show()

fig = px.histogram(det, x='dias_para_ofertar', nbins=25, title='Días entre publicación y cierre de ofertas',
                   labels={'dias_para_ofertar': 'días'}, color_discrete_sequence=['#228054'])
fig.show()

In [ ]:
montos = det[(det['monto_estimado'].fillna(0) > 0) & (det['moneda'] == 'CLP')]
print(f'{len(montos)} licitaciones de la muestra publican monto estimado en CLP')
if len(montos):
    fig = px.box(montos, y='monto_estimado', points='all', hover_name='nombre', log_y=True,
                 title='Monto estimado publicado (CLP, escala log)')
    fig.show()
resumen = pd.Series({
    'Licitaciones analizadas': len(df),
    'Con detalle': len(det),
    'Mediana de días para ofertar': det['dias_para_ofertar'].median(),
    '% que menciona garantía': round(100 * det['menciona_garantia'].mean(), 1),
    '% que menciona multas': round(100 * det['menciona_multa'].mean(), 1),
})
resumen.to_frame('valor')

### 2.8 Nube de palabras vs. embeddings: la comparación clave
La nube trata cada palabra por separado. Los embeddings agrupan por significado. Busquemos licitaciones de **aseo y limpieza** con ambos enfoques.

In [ ]:
consulta = 'artículos de aseo y limpieza'
q_tfidf = tfidf.transform([limpiar(consulta)]).toarray()[0]
q_tfidf = q_tfidf / max(np.linalg.norm(q_tfidf), 1e-9)
r_tfidf = df['nombre'].iloc[np.argsort(-(Xt_n @ q_tfidf))[:8]].tolist()

try:
    q_emb = cargar_modelo().encode([consulta], normalize_embeddings=True)[0]
    r_emb = df['nombre'].iloc[np.argsort(-(E @ q_emb))[:8]].tolist()
except Exception as e:
    print('Modelo no disponible en este entorno (en Colab sí funciona).'); r_emb = ['—'] * 8

pd.DataFrame({'Con TF-IDF (palabras)': r_tfidf, 'Con embeddings (significado)': r_emb})

## Preguntas para el informe
1. ¿Qué agrega cada gráfico clásico (nube, barras, n-gramas, red, LDA) que no muestra el anterior?
2. Compara los temas de **LDA** y de **BERTopic**: ¿cuáles son más coherentes y por qué?
3. Encuentra en el mapa de embeddings dos licitaciones cercanas que **no compartan palabras**. ¿Qué tienen en común?
4. Diseña 5 campos adicionales para el esquema JSON que le servirían a una empresa de tu rubro para decidir si postular.
5. ¿Qué campo extraído exige mayor *recall* y por qué?

---
## Anexo A · Descargar licitaciones desde la API de Mercado Público
Solicita tu propio ticket en <https://api.mercadopublico.cl> (gratuito). La API limita la cantidad de consultas por minuto, por eso se espera entre llamadas.

In [ ]:
import time, requests
TICKET = ''           # <- pega aquí tu ticket
FECHA = '06102026'    # formato ddmmaaaa

if TICKET:
    url = 'https://api.mercadopublico.cl/servicios/v1/publico/licitaciones.json'
    lista = requests.get(url, params={'fecha': FECHA, 'ticket': TICKET}, timeout=60).json()['Listado']
    print(len(lista), 'licitaciones el', FECHA)
    # Detalle de una licitación
    time.sleep(2)
    detalle = requests.get(url, params={'codigo': lista[0]['CodigoExterno'], 'ticket': TICKET}, timeout=60).json()['Listado'][0]
    print(json.dumps({k: detalle[k] for k in ['Nombre', 'Descripcion', 'MontoEstimado']}, ensure_ascii=False, indent=2))
else:
    print('Agrega tu ticket para consultar la API.')